# LieWaves — Domain Adaptation + Explainability (Grad-CAM)
### Forschungsfragen:
- **RQ1:** Kann Domain Adaptation (DANN) den Subject Gap schließen?
- **RQ2:** Lernt das Modell wirklich neurophysiologisch sinnvolle Merkmale (P300, frontales Theta)?

**Unterschied zum Original-Notebook:**
| Original | Dieses Notebook |
|---|---|
| Random 80/20 Split (subject-dependent) | Leave-One-Subject-Out (LOSO) |
| Einfaches CNN | DANN mit Gradient Reversal Layer |
| Keine Erklärbarkeit | Grad-CAM Visualisierung |
| DWT Features (flach) | DWT Features (strukturiert, pro Subject) |

## Schritt 0: Installation

In [ ]:
%pip install scikit-learn PyWavelets tensorflow numpy pandas matplotlib seaborn -q

## Schritt 1: Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras.layers import Layer
import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices("GPU")) > 0)

## Schritt 2: Daten laden — pro Subject

**Wichtig:** Für LOSO brauchen wir die Daten **getrennt pro Subject**, nicht alles zusammen wie im Original.

Erwartet wird folgende Ordnerstruktur:
```
Truth_Sessions/1_BandPass_Filtered/subject_01.csv
Truth_Sessions/1_BandPass_Filtered/subject_02.csv
...
Lie_Sessions/1_BandPass_Filtered/subject_01.csv
...
```
Jede CSV enthält 5 Spalten (Kanäle: AF3, AF4, T7, T8, Pz).

In [ ]:
# ── Pfade anpassen ──────────────────────────────────────────
TRUTH_PATH = '../LieWaves/Truth_Sessions/1_BandPass_Filtered/'
LIE_PATH   = '../LieWaves/Lie_Sessions/1_BandPass_Filtered/'

WINDOW_SIZE = 128   # 1 Sekunde bei 128 Hz
OVERLAP     = 64    # 50% Overlap
WAVELET     = 'db4'
LEVEL       = 4
CHANNELS    = ['AF3', 'AF4', 'T7', 'T8', 'Pz']
# ────────────────────────────────────────────────────────────

## Schritt 3: Gradient Reversal Layer (Kernstück des DANN)

Die GRL ist das mathematische Herzstück der Domain Adaptation:
- **Forward Pass:** Identitätsfunktion (keine Änderung)
- **Backward Pass:** Gradient wird mit `-λ` multipliziert

→ Der Feature Extractor lernt, Merkmale zu erzeugen, die **gut für Lügenerkennung** aber **schlecht für Subject-Unterscheidung** sind.

In [ ]:
@tf.custom_gradient
def _gradient_reverse(x):
    """Identität vorwärts, negierter Gradient rückwärts."""
    def grad(dy):
        return -dy
    return x, grad

class GradientReversalLayer(Layer):
    """
    Gradient Reversal Layer nach Ganin & Lempitsky (2015).
    λ (lambda_) bestimmt die Stärke der Domain-Adversarial-Regularisierung.
    Wird typischerweise während des Trainings von 0 → 1 angehoben.
    """
    def __init__(self, lambda_=1.0, **kwargs):
        super().__init__(**kwargs)
        self.lambda_ = tf.Variable(lambda_, trainable=False, dtype=tf.float32)

    def call(self, x):
        return self.lambda_ * _gradient_reverse(x)

    def get_config(self):
        config = super().get_config()
        config.update({'lambda_': float(self.lambda_.numpy())})
        return config

print('✅ Gradient Reversal Layer definiert.')

## Schritt 4: DANN Modell-Architektur

```
Input (DWT Features)
       │
  ─────────────────
  | Feature       |
  | Extractor     |  ← Gemeinsamer Backbone (LSTM)
  | (geteilt)     |
  ─────────────────
       │
  ┌────┴────────────────────┐
  │                         │
Label Classifier      GRL (−λ)
  │                         │
  │                   Domain Classifier
  │                         │
Truth/Lie             Welches Subject?
(Hauptaufgabe)        (Adversarisch)
```

In [ ]:
def build_dann(input_dim: int, n_subjects: int, lambda_: float = 1.0):
    """
    Baut das Domain Adversarial Neural Network.
    
    Args:
        input_dim:  Anzahl DWT Features
        n_subjects: Anzahl Subjects (für Domain Classifier)
        lambda_:    GRL Stärke (wird im Training angepasst)
    """
    inputs = tf.keras.Input(shape=(input_dim,), name='features')

    # ── Feature Extractor ─────────────────────────────────────
    x = layers.Dense(256, activation='relu', name='fe_dense1')(inputs)
    x = layers.BatchNormalization(name='fe_bn1')(x)
    x = layers.Dropout(0.3, name='fe_drop1')(x)

    x = layers.Dense(128, activation='relu', name='fe_dense2')(x)
    x = layers.BatchNormalization(name='fe_bn2')(x)
    x = layers.Dropout(0.3, name='fe_drop2')(x)

    features = layers.Dense(64, activation='relu', name='feature_space')(x)

    # ── Label Classifier (Truth vs Lie) ───────────────────────
    lc = layers.Dense(32, activation='relu', name='lc_dense1')(features)
    lc = layers.Dropout(0.2, name='lc_drop')(lc)
    label_output = layers.Dense(1, activation='sigmoid', name='label_output')(lc)

    # ── Domain Classifier (welches Subject?) ──────────────────
    grl = GradientReversalLayer(lambda_=lambda_, name='grl')(features)
    dc  = layers.Dense(32, activation='relu', name='dc_dense1')(grl)
    dc  = layers.Dropout(0.2, name='dc_drop')(dc)
    domain_output = layers.Dense(n_subjects, activation='softmax', name='domain_output')(dc)

    model = tf.keras.Model(
        inputs=inputs,
        outputs={'label_output': label_output, 'domain_output': domain_output},
        name='DANN'
    )
    return model

# Testweise bauen und zusammenfassen
n_features  = all_features[0].shape[1]
n_subjects  = len(subject_ids)

demo_model = build_dann(n_features, n_subjects)
demo_model.summary(line_length=70)

## Schritt 5: LOSO Training (Leave-One-Subject-Out)

**Warum LOSO?**
- Subject i = **Test Set**
- Alle anderen 26 Subjects = **Training Set**
- 27 Durchläufe insgesamt → realistische Schätzung der Generalisierung auf unbekannte Personen

**Wichtig:** Der GRL-Lambda wird während des Trainings von 0 → 1 angehoben (Warm-up Strategie).

In [ ]:
class LambdaScheduler(callbacks.Callback):
    """Hebt GRL-Lambda schrittweise von 0 auf lambda_max an."""
    def __init__(self, model, lambda_max=1.0, total_epochs=50):
        super().__init__()
        self.grl         = model.get_layer('grl')
        self.lambda_max  = lambda_max
        self.total_epochs = total_epochs

    def on_epoch_begin(self, epoch, logs=None):
        p = epoch / self.total_epochs
        new_lambda = float(self.lambda_max * (2 / (1 + np.exp(-10 * p)) - 1))
        self.grl.lambda_.assign(new_lambda)


def run_loso(all_features, all_labels, subject_ids,
             epochs=50, batch_size=64, verbose=0):
    """
    Führt vollständige LOSO-Evaluation mit DANN durch.
    Vergleicht außerdem mit einem einfachen Baseline-Modell (kein DA).
    """
    results = []
    n_subj  = len(subject_ids)

    for fold, test_idx in enumerate(range(n_subj)):
        print(f'\n── Fold {fold+1}/{n_subj}: Test-Subject = {subject_ids[test_idx]} ──')

        # ── Daten aufteilen ───────────────────────────────────
        X_test  = all_features[test_idx]
        y_test  = all_labels[test_idx]
        d_test  = np.full(len(y_test), test_idx)  # Domain-Label (ignoriert beim Test)

        train_idxs = [i for i in range(n_subj) if i != test_idx]
        X_train = np.vstack([all_features[i] for i in train_idxs])
        y_train = np.hstack([all_labels[i]   for i in train_idxs])
        # Domain-Labels: 0..n_subj-1 (test_idx ausgelassen)
        d_train = np.hstack([
            np.full(len(all_labels[i]), j)
            for j, i in enumerate(train_idxs)
        ])

        # ── Normalisierung auf Training-Statistiken ───────────
        scaler  = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        # ── DANN bauen & trainieren ───────────────────────────
        model = build_dann(X_train.shape[1], n_subjects=n_subj - 1)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-3),
            loss={
                'label_output':  'binary_crossentropy',
                'domain_output': 'sparse_categorical_crossentropy'
            },
            loss_weights={'label_output': 1.0, 'domain_output': 0.5},
            metrics={'label_output': 'accuracy'}
        )

        cb = [
            LambdaScheduler(model, lambda_max=1.0, total_epochs=epochs),
            callbacks.EarlyStopping(
                monitor='val_label_output_accuracy', patience=8,
                restore_best_weights=True, mode='max'
            )
        ]

        model.fit(
            X_train,
            {'label_output': y_train, 'domain_output': d_train},
            validation_split=0.1,
            epochs=epochs, batch_size=batch_size,
            callbacks=cb, verbose=verbose
        )

        # ── Evaluation ────────────────────────────────────────
        preds   = model.predict(X_test, verbose=0)
        y_prob  = preds['label_output'].flatten()
        y_pred  = (y_prob > 0.5).astype(int)

        acc  = np.mean(y_pred == y_test)
        f1   = f1_score(y_test, y_pred, zero_division=1)
        try:
            auc = roc_auc_score(y_test, y_prob)
        except Exception:
            auc = float('nan')

        results.append({
            'subject': subject_ids[test_idx],
            'accuracy': acc, 'f1': f1, 'auc': auc,
            'model': model, 'scaler': scaler,
            'X_test': X_test, 'y_test': y_test
        })
        print(f'   Accuracy: {acc:.3f}  |  F1: {f1:.3f}  |  AUC: {auc:.3f}')

    return results


# ── Training starten ──────────────────────────────────────────
# Tipp: Für schnellen Test erst mit 2-3 Subjects probieren:
# loso_results = run_loso(all_features[:3], all_labels[:3], subject_ids[:3])

loso_results = run_loso(all_features, all_labels, subject_ids, epochs=50)

## Schritt 6: Ergebnisse — LOSO Übersicht

In [ ]:
# ── Zusammenfassung ───────────────────────────────────────────
df_results = pd.DataFrame([{
    'Subject':  r['subject'],
    'Accuracy': round(r['accuracy'], 4),
    'F1 Score': round(r['f1'], 4),
    'ROC-AUC':  round(r['auc'], 4)
} for r in loso_results])

print('=' * 50)
print('LOSO Ergebnisse — DANN (Subject-Independent)')
print('=' * 50)
print(df_results.to_string(index=False))
print('─' * 50)
print(f"Ø Accuracy: {df_results['Accuracy'].mean():.4f} ± {df_results['Accuracy'].std():.4f}")
print(f"Ø F1 Score: {df_results['F1 Score'].mean():.4f} ± {df_results['F1 Score'].std():.4f}")
print(f"Ø ROC-AUC:  {df_results['ROC-AUC'].mean():.4f}  ± {df_results['ROC-AUC'].std():.4f}")

# ── Visualisierung: Accuracy pro Subject ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Balkendiagramm Accuracy
colors = ['#2E75B6' if a >= 0.7 else '#E74C3C' for a in df_results['Accuracy']]
axes[0].bar(df_results['Subject'], df_results['Accuracy'], color=colors)
axes[0].axhline(df_results['Accuracy'].mean(), color='black', linestyle='--',
                label=f"Ø = {df_results['Accuracy'].mean():.3f}")
axes[0].axhline(0.5, color='gray', linestyle=':', label='Zufallslevel')
axes[0].set_xlabel('Subject')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('LOSO Accuracy pro Subject (DANN)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylim([0, 1.05])

# Metriken Boxplot
axes[1].boxplot(
    [df_results['Accuracy'], df_results['F1 Score'], df_results['ROC-AUC']],
    labels=['Accuracy', 'F1 Score', 'ROC-AUC'],
    patch_artist=True,
    boxprops=dict(facecolor='#D6E4F0'),
    medianprops=dict(color='#2E75B6', linewidth=2)
)
axes[1].set_title('Verteilung der Metriken über alle Subjects')
axes[1].set_ylim([0, 1.05])
axes[1].axhline(0.5, color='gray', linestyle=':', label='Zufallslevel')
axes[1].legend()

plt.tight_layout()
plt.savefig('loso_results.png', dpi=150)
plt.show()
print('💾 Grafik gespeichert: loso_results.png')

## Schritt 7: Baseline-Vergleich (Subject-Dependent, wie Original-Notebook)

Wichtig für die Forschungsfrage: **Wie groß ist der Subject Gap?**

> Subject Gap = Accuracy (subject-dependent) − Accuracy (LOSO)

In [ ]:
from sklearn.model_selection import StratifiedKFold

def run_baseline_within_subject(all_features, all_labels, subject_ids, n_splits=5):
    """Einfaches CNN, 5-Fold CV innerhalb jedes Subjects (subject-dependent)."""
    results = []
    for i, (X, y) in enumerate(zip(all_features, all_labels)):
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        fold_accs = []
        for train_idx, val_idx in skf.split(X, y):
            m = models.Sequential([
                layers.Dense(128, activation='relu', input_shape=(X.shape[1],)),
                layers.Dropout(0.3),
                layers.Dense(64, activation='relu'),
                layers.Dense(1, activation='sigmoid')
            ])
            m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
            m.fit(X[train_idx], y[train_idx], epochs=20, batch_size=32, verbose=0)
            _, acc = m.evaluate(X[val_idx], y[val_idx], verbose=0)
            fold_accs.append(acc)
        results.append({'subject': subject_ids[i], 'accuracy_within': np.mean(fold_accs)})
        print(f'  Subject {subject_ids[i]}: Within-Subject Accuracy = {np.mean(fold_accs):.3f}')
    return pd.DataFrame(results)

print('Berechne Subject-Dependent Baseline...')
df_baseline = run_baseline_within_subject(all_features, all_labels, subject_ids)

# ── Subject Gap berechnen ─────────────────────────────────────
df_gap = df_baseline.merge(
    df_results[['Subject', 'Accuracy']].rename(columns={'Subject': 'subject', 'Accuracy': 'accuracy_loso'}),
    on='subject'
)
df_gap['subject_gap'] = df_gap['accuracy_within'] - df_gap['accuracy_loso']

print('\n' + '=' * 55)
print('Subject Gap Analyse')
print('=' * 55)
print(df_gap[['subject', 'accuracy_within', 'accuracy_loso', 'subject_gap']].to_string(index=False))
print(f"\nØ Subject Gap: {df_gap['subject_gap'].mean():.4f}")
print(f"→ Das Modell verliert im Schnitt {df_gap['subject_gap'].mean()*100:.1f}% Genauigkeit bei unbekannten Personen.")

# ── Visualisierung ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(df_gap))
w = 0.35
ax.bar(x - w/2, df_gap['accuracy_within'], w, label='Within-Subject (Baseline)', color='#2E75B6')
ax.bar(x + w/2, df_gap['accuracy_loso'],   w, label='LOSO — DANN',              color='#27AE60')
ax.set_xticks(x)
ax.set_xticklabels(df_gap['subject'], rotation=45)
ax.set_ylabel('Accuracy')
ax.set_title('Subject Gap: Within-Subject vs. LOSO (DANN)')
ax.legend()
ax.set_ylim([0, 1.1])
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('subject_gap.png', dpi=150)
plt.show()
print('💾 Grafik gespeichert: subject_gap.png')

## Schritt 8: Grad-CAM — Was lernt das Modell?

**Ziel (RQ2):** Prüfen, ob das Modell auf neurophysiologisch sinnvolle Features fokussiert:
- **P300** → parietale Kanäle (Pz), 300–500ms nach Stimulus
- **Frontales Theta** → AF3/AF4, 4–8 Hz
- **T7/T8** → temporale Kanäle (besonders wichtig laut LieWaves-Paper)

Hier: Grad-CAM auf den Feature Extractor des DANN.

In [ ]:
def compute_gradcam_feature_importance(model, X_samples, layer_name='feature_space'):
    """
    Berechnet Grad-CAM Feature-Wichtigkeiten für das Label-Output.
    Gibt den Durchschnitt der absoluten Gradienten über alle Samples zurück.
    """
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output,
                 model.output['label_output']]
    )

    all_grads = []
    for i in range(0, len(X_samples), 32):   # Batch-weise für Speichereffizienz
        batch = tf.constant(X_samples[i:i+32], dtype=tf.float32)
        with tf.GradientTape() as tape:
            tape.watch(batch)
            feature_out, label_out = grad_model(batch, training=False)
            # Gradient bezüglich des Label-Outputs
            loss = tf.reduce_mean(label_out)
        grads = tape.gradient(loss, batch)    # Gradient w.r.t. Input-Features
        all_grads.append(np.abs(grads.numpy()))

    # Durchschnittliche Wichtigkeit pro Feature-Dimension
    importance = np.mean(np.vstack(all_grads), axis=0)  # (n_features,)
    return importance


def plot_gradcam_per_channel(importance, wavelet=WAVELET, level=LEVEL,
                              channels=CHANNELS, subject_name=''):
    """
    Visualisiert Grad-CAM Wichtigkeiten aufgeteilt nach EEG-Kanal.
    Pro Kanal: 5 Sub-Bänder × 5 Statistiken = 25 Features.
    """
    n_subbands    = level + 1   # cA + cD1..cD4
    n_stats       = 5           # mean, std, energy, kurtosis, skewness
    feats_per_ch  = n_subbands * n_stats
    stat_names    = ['Mean', 'Std', 'Energy', 'Kurtosis', 'Skewness']
    band_names    = ['cA (Approx)'] + [f'cD{i+1}' for i in range(level)]

    fig, axes = plt.subplots(1, len(channels), figsize=(20, 5), sharey=False)
    fig.suptitle(f'Grad-CAM Feature-Wichtigkeit pro EEG-Kanal\n'
                 f'Subject: {subject_name}  |  Wavelet: {wavelet}, Level: {level}',
                 fontsize=13, fontweight='bold')

    vmax = importance.max()

    for ci, (ch, ax) in enumerate(zip(channels, axes)):
        start = ci * feats_per_ch
        ch_imp = importance[start:start + feats_per_ch].reshape(n_subbands, n_stats)

        im = ax.imshow(ch_imp, aspect='auto', cmap='YlOrRd', vmin=0, vmax=vmax)
        ax.set_title(f'Kanal: {ch}', fontweight='bold')
        ax.set_xticks(range(n_stats))
        ax.set_xticklabels(stat_names, rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(n_subbands))
        ax.set_yticklabels(band_names, fontsize=8)
        plt.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()
    fname = f'gradcam_{subject_name}.png'
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f'💾 Gespeichert: {fname}')


# ── Grad-CAM für den ersten LOSO-Fold berechnen ───────────────
fold = loso_results[0]  # erstes Subject als Beispiel

print(f'Berechne Grad-CAM für Subject: {fold["subject"]}...')
importance = compute_gradcam_feature_importance(fold['model'], fold['X_test'])

plot_gradcam_per_channel(importance, subject_name=fold['subject'])

# ── Top Features ausgeben ─────────────────────────────────────
top_k = 10
top_idx = np.argsort(importance)[::-1][:top_k]
print(f'\nTop {top_k} wichtigste Feature-Dimensionen (Index → Wichtigkeit):')
for rank, idx in enumerate(top_idx):
    ch_idx   = idx // ((LEVEL + 1) * 5)
    rest     = idx %  ((LEVEL + 1) * 5)
    band_idx = rest // 5
    stat_idx = rest %  5
    ch_name   = CHANNELS[ch_idx] if ch_idx < len(CHANNELS) else f'Ch{ch_idx}'
    band_name = ['cA']+[f'cD{i+1}' for i in range(LEVEL)]
    band_name = band_name[band_idx] if band_idx < len(band_name) else f'B{band_idx}'
    stat_name = ['Mean','Std','Energy','Kurtosis','Skewness'][stat_idx]
    print(f'  #{rank+1:2d}: {ch_name:5s} | {band_name:10s} | {stat_name:10s} → {importance[idx]:.6f}')

## Schritt 9: Aggregierter Grad-CAM über alle Subjects

Zeigt, welche Kanäle und Sub-Bänder **konsistent** über alle 27 Subjects wichtig sind — das ist die neurophysiologische Validierung.

In [ ]:
print('Berechne Grad-CAM für alle Subjects...')
all_importances = []

for fold in loso_results:
    imp = compute_gradcam_feature_importance(fold['model'], fold['X_test'])
    all_importances.append(imp)
    print(f'  ✅ {fold["subject"]}')

mean_importance = np.mean(all_importances, axis=0)
std_importance  = np.std(all_importances,  axis=0)

# ── Kanal-Aggregation ─────────────────────────────────────────
feats_per_ch = (LEVEL + 1) * 5
channel_importance = [
    mean_importance[i * feats_per_ch:(i + 1) * feats_per_ch].sum()
    for i in range(len(CHANNELS))
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kanal-Wichtigkeit
colors = ['#E74C3C' if ch in ['T7', 'T8'] else '#2E75B6' for ch in CHANNELS]
axes[0].bar(CHANNELS, channel_importance, color=colors)
axes[0].set_title('Aggregierte Grad-CAM Wichtigkeit pro Kanal\n'
                  '(rot = temporale Kanäle T7/T8 — laut Literatur besonders relevant)')
axes[0].set_ylabel('Kumulative Grad-CAM Wichtigkeit')

# Heatmap: Kanal × Sub-Band
band_names = ['cA'] + [f'cD{i+1}' for i in range(LEVEL)]
importance_matrix = mean_importance.reshape(len(CHANNELS), LEVEL + 1, 5).mean(axis=2)
im = axes[1].imshow(importance_matrix, cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(LEVEL + 1))
axes[1].set_xticklabels(band_names)
axes[1].set_yticks(range(len(CHANNELS)))
axes[1].set_yticklabels(CHANNELS)
axes[1].set_title('Grad-CAM Heatmap: Kanal × Frequenzband\n(Durchschnitt über alle Subjects)')
axes[1].set_xlabel('DWT Sub-Band')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig('gradcam_aggregated.png', dpi=150)
plt.show()

print('\n── Interpretation ──────────────────────────────────────────')
top_ch = CHANNELS[np.argmax(channel_importance)]
print(f'Wichtigster Kanal: {top_ch}')
print('Erwartung aus Literatur: T7 oder T8 (temporal) oder Pz (parietal/P300)')
if top_ch in ['T7', 'T8']:
    print('✅ Konsistent mit LieWaves-Paper: temporale Kanäle dominieren.')
elif top_ch == 'Pz':
    print('✅ Konsistent mit P300-Literatur: parietaler Kanal dominiert.')
else:
    print('⚠️  Unerwartetes Ergebnis — könnte auf Artefakte hinweisen.')

## Schritt 10: Finale Zusammenfassung & Diskussion

In [ ]:
print('=' * 60)
print('FINALE ZUSAMMENFASSUNG — LieWaves DANN + Grad-CAM')
print('=' * 60)

within_acc = df_gap['accuracy_within'].mean()
loso_acc   = df_gap['accuracy_loso'].mean()
gap        = df_gap['subject_gap'].mean()

print(f"\n{'Metrik':<35} {'Wert':>10}")
print('-' * 47)
print(f"{'Within-Subject Accuracy (Baseline)':<35} {within_acc:>9.4f}")
print(f"{'LOSO Accuracy (DANN — subject-indep.)':<35} {loso_acc:>9.4f}")
print(f"{'Subject Gap (Verlust durch LOSO)':<35} {gap:>9.4f}")
print(f"{'Ø F1 Score (LOSO)':<35} {df_results['F1 Score'].mean():>9.4f}")
print(f"{'Ø ROC-AUC (LOSO)':<35} {df_results['ROC-AUC'].mean():>9.4f}")

print('\n── RQ1: Domain Adaptation ──────────────────────────────────')
if loso_acc > 0.70:
    print(f'✅ DANN erreicht {loso_acc:.1%} im LOSO — solide subject-independent Performance.')
elif loso_acc > 0.60:
    print(f'⚠️  DANN erreicht {loso_acc:.1%} — besser als Zufall, aber Subject Gap bleibt.')
else:
    print(f'❌ DANN erreicht nur {loso_acc:.1%} — Subject Gap ist gravierend.')
print(f'   → Original LieWaves (within-subject): ~99.88%')
print(f'   → Diese Arbeit (LOSO/DANN):           {loso_acc:.2%}')
print(f'   → Subject Gap:                         {gap:.2%} Punkte')

print('\n── RQ2: Explainability (Grad-CAM) ─────────────────────────')
top_ch = CHANNELS[np.argmax(channel_importance)]
print(f'   Wichtigster Kanal: {top_ch}')
print(f'   → Prüfen: stimmt das mit P300 / frontalem Theta überein?')
print(f'   → Falls T7/T8 dominant: konsistent mit LieWaves-Ergebnis ✅')
print(f'   → Falls AF3/AF4 dominant: konsistent mit frontalem Theta ✅')

print('\n── Nächste Schritte ────────────────────────────────────────')
print('  1. Wavelet-Vergleich: db4 vs. sym4 vs. coif3 (RQ3)')
print('  2. Cross-Dataset: Modell auf Bag-of-Lies testen')
print('  3. Mehrere Subjects mit Psychometrie-Daten korrelieren')
print('=' * 60)